### Polars: современная альтернатива pandas

In [ ]:
import time
import numpy as np
import pandas as pd
import polars as pl

print("pandas:", pd.__version__, "| polars:", pl.__version__)

n = 10_000_000
rng = np.random.default_rng(42)
данные = {
    "значение":  rng.standard_normal(n),
    "категория": rng.choice(["A", "B", "C", "D"], n),
}
df_pd = pd.DataFrame(данные)

t0 = time.perf_counter()
res_pd = df_pd.groupby("категория")["значение"].agg(["mean", "std", "count"])
t_pd = time.perf_counter() - t0
print(f"pandas groupby: {t_pd:.2f}с")   # типично ~0.8–1.5с на ноутбуке
res_pd

In [ ]:
df_pl_big = pl.DataFrame(данные)

t0 = time.perf_counter()
res_pl = df_pl_big.group_by("категория").agg([
    pl.col("значение").mean().alias("mean"),
    pl.col("значение").std().alias("std"),
    pl.col("значение").count().alias("count"),
]).sort("категория")
t_pl = time.perf_counter() - t0
print(f"Polars group_by: {t_pl:.2f}с  (в {t_pd / t_pl:.1f} раз быстрее)")
res_pl

### Использование ядер

In [3]:
import os
import time
import numpy as np
import pandas as pd
import polars as pl


N = 20_000_000
N_KEYS = 10_000

rng = np.random.default_rng(1)
data = {
    "v": rng.standard_normal(N),
    "k": rng.choice([f"K{i}" for i in range(N_KEYS)], N),
}
df_pd = pd.DataFrame(data)
df_pl = pl.DataFrame(data)


def cores_used(fn, *, warmup=1, repeats=5):
    for _ in range(warmup):
        fn()

    wall0, cpu0 = time.perf_counter(), time.process_time()
    for _ in range(repeats):
        fn()
    wall = time.perf_counter() - wall0
    cpu = time.process_time() - cpu0

    return cpu / wall if wall > 0 else float("nan")


def report(label, fn):
    cores = cores_used(fn)
    print(f"{label:<10} {cores:6.2f} cores  (of {os.cpu_count()} available)")


pandas_std = lambda: df_pd.groupby("k")["v"].std()
polars_std = lambda: df_pl.group_by("k").agg(pl.col("v").std())

report("pandas:", pandas_std)
report("Polars:", polars_std)

pandas:      0.62 cores  (of 8 available)
Polars:      1.30 cores  (of 8 available)


### Eager vs Lazy API и выражения

In [4]:
df = pl.DataFrame({
    "страна":    ["Германия", "Франция", "Италия", "Испания"],
    "ввп":       [4.07, 2.78, 2.10, 1.77],
    "инфляция":  [2.1, 2.5, 3.0, 3.5],
    "население": [83.2, 67.8, 59.6, 47.4],
})

print(df)
print(df.schema)   # типы данных (аналог dtypes)
print(df.shape)
df.head(2)

shape: (4, 4)
┌──────────┬──────┬──────────┬───────────┐
│ страна   ┆ ввп  ┆ инфляция ┆ население │
│ ---      ┆ ---  ┆ ---      ┆ ---       │
│ str      ┆ f64  ┆ f64      ┆ f64       │
╞══════════╪══════╪══════════╪═══════════╡
│ Германия ┆ 4.07 ┆ 2.1      ┆ 83.2      │
│ Франция  ┆ 2.78 ┆ 2.5      ┆ 67.8      │
│ Италия   ┆ 2.1  ┆ 3.0      ┆ 59.6      │
│ Испания  ┆ 1.77 ┆ 3.5      ┆ 47.4      │
└──────────┴──────┴──────────┴───────────┘
Schema({'страна': String, 'ввп': Float64, 'инфляция': Float64, 'население': Float64})
(4, 4)


страна,ввп,инфляция,население
str,f64,f64,f64
"""Германия""",4.07,2.1,83.2
"""Франция""",2.78,2.5,67.8


### Eager API - выполнение немедленно

In [5]:
# Выбор столбцов
df.select(["страна", "ввп"])

страна,ввп
str,f64
"""Германия""",4.07
"""Франция""",2.78
"""Италия""",2.1
"""Испания""",1.77


In [6]:
df.select(pl.col("ввп"), pl.col("инфляция"))

ввп,инфляция
f64,f64
4.07,2.1
2.78,2.5
2.1,3.0
1.77,3.5


In [7]:
# Фильтрация
df.filter(pl.col("ввп") > 2.5)

страна,ввп,инфляция,население
str,f64,f64,f64
"""Германия""",4.07,2.1,83.2
"""Франция""",2.78,2.5,67.8


In [8]:
df.filter((pl.col("ввп") > 2.0) & (pl.col("инфляция") < 3.0))

страна,ввп,инфляция,население
str,f64,f64,f64
"""Германия""",4.07,2.1,83.2
"""Франция""",2.78,2.5,67.8


In [9]:
# Добавление столбца (with_columns - аналог assign в pandas)
df.with_columns(
    (pl.col("ввп") / pl.col("население") * 1000).alias("ввп_на_душу")
)

страна,ввп,инфляция,население,ввп_на_душу
str,f64,f64,f64,f64
"""Германия""",4.07,2.1,83.2,48.918269
"""Франция""",2.78,2.5,67.8,41.00295
"""Италия""",2.1,3.0,59.6,35.234899
"""Испания""",1.77,3.5,47.4,37.341772


In [ ]:
# Сортировка
df.sort("ввп", descending=True)

In [ ]:
# Переименование
df.rename({"ввп": "gdp", "инфляция": "inflation"})

### Lazy API - выполнение только при `.collect()`
`df.lazy()` строит *план* запроса, ничего не вычисляя. Polars оптимизирует план и выполняет его целиком.

In [10]:
lazy = df.lazy()

query = (
    lazy
    .filter(pl.col("ввп") > 2.0)
    .with_columns(
        (pl.col("ввп") / pl.col("население")).alias("ввп_на_душу")
    )
    .sort("ввп_на_душу", descending=True)
    .select(["страна", "ввп", "ввп_на_душу"])
)

print(type(query))        # LazyFrame - ничего ещё не посчитано
print(query.explain())    # план выполнения

<class 'polars.lazyframe.frame.LazyFrame'>
SORT BY [descending: [true]] [col("ввп_на_душу")]
  simple π 3/3 ["страна", "ввп", ... 1 other column]
     WITH_COLUMNS:
     [[(col("ввп")) / (col("население"))].alias("ввп_на_душу")] 
      FILTER [(col("ввп")) > (2.0)]
      FROM
        DF ["страна", "ввп", "инфляция", "население"]; PROJECT["ввп", "население", "страна"] 3/4 COLUMNS


In [11]:
result = query.collect()   # выполнить!
result

страна,ввп,ввп_на_душу
str,f64,f64
"""Германия""",4.07,0.048918
"""Франция""",2.78,0.041003
"""Италия""",2.1,0.035235


### API выражений - сердце Polars

In [12]:
rng = np.random.default_rng(0)
m = 1000
sales = pl.DataFrame({
    "дата":      [f"2024-{rng.integers(1, 13):02d}-{rng.integers(1, 29):02d}" for _ in range(m)],
    "регион":    rng.choice(["Север", "Юг", "Запад", "Восток"], m),
    "категория": rng.choice(["Электроника", "Одежда", "Книги", "Еда"], m),
    "клиент":    rng.integers(1, 120, m),
    "сумма":     np.round(rng.gamma(2.0, 400.0, m), 2),
})
sales.write_csv("../data/sales.csv")
print(open("../data/sales.csv", encoding="utf-8").read()[:250])

дата,регион,категория,клиент,сумма
2024-11-18,Запад,Электроника,49,1939.32
2024-07-08,Север,Электроника,117,563.93
2024-04-02,Запад,Одежда,77,721.17
2024-01-01,Восток,Книги,8,696.66
2024-03-23,Восток,Книги,94,440.59
2024-08-26,Юг,Электроника,110,60.1


In [13]:
df = pl.read_csv("../data/sales.csv")

(df.lazy()
    .with_columns([
        # Арифметика
        (pl.col("сумма") * 1.2).alias("сумма_с_ндс"),
        # Строковые операции
        pl.col("категория").str.to_lowercase().alias("кат_lower"),
        # Условие (аналог np.where)
        pl.when(pl.col("сумма") > 1000)
          .then(pl.lit("крупная"))
          .otherwise(pl.lit("малая"))
          .alias("размер"),
        # Дата
        pl.col("дата").str.to_date().alias("дата_dt"),
    ])
    .collect()
    .head(5)
)

дата,регион,категория,клиент,сумма,сумма_с_ндс,кат_lower,размер,дата_dt
str,str,str,i64,f64,f64,str,str,date
"""2024-11-18""","""Запад""","""Электроника""",49,1939.32,2327.184,"""электроника""","""крупная""",2024-11-18
"""2024-07-08""","""Север""","""Электроника""",117,563.93,676.716,"""электроника""","""малая""",2024-07-08
"""2024-04-02""","""Запад""","""Одежда""",77,721.17,865.404,"""одежда""","""малая""",2024-04-02
"""2024-01-01""","""Восток""","""Книги""",8,696.66,835.992,"""книги""","""малая""",2024-01-01
"""2024-03-23""","""Восток""","""Книги""",94,440.59,528.708,"""книги""","""малая""",2024-03-23


In [14]:
# Выбор столбцов по типу - применить ко всем Float64-столбцам
(df.lazy()
    .with_columns(pl.col(pl.Float64).round(1))
    .collect()
    .head(5)
)

дата,регион,категория,клиент,сумма
str,str,str,i64,f64
"""2024-11-18""","""Запад""","""Электроника""",49,1939.3
"""2024-07-08""","""Север""","""Электроника""",117,563.9
"""2024-04-02""","""Запад""","""Одежда""",77,721.2
"""2024-01-01""","""Восток""","""Книги""",8,696.7
"""2024-03-23""","""Восток""","""Книги""",94,440.6


In [16]:
df.to_pandas()

,дата,регион,категория,клиент,сумма
0,2024-11-18,Запад,Электроника,49,1939.32
1,2024-07-08,Север,Электроника,117,563.93
2,2024-04-02,Запад,Одежда,77,721.17
3,2024-01-01,Восток,Книги,8,696.66
4,2024-03-23,Восток,Книги,94,440.59
...,...,...,...,...,...
995,2024-07-24,Восток,Книги,89,831.99
996,2024-09-03,Запад,Одежда,100,187.53
997,2024-02-11,Восток,Электроника,63,202.23
998,2024-01-05,Север,Электроника,33,654.95


### Сравнение синтаксиса pandas vs Polars

In [ ]:
df_pd = df.to_pandas()
df_pl = df

# ФИЛЬТРАЦИЯ
r1 = df_pd[df_pd["сумма"] > 500]                                  # pandas
r2 = df_pl.filter(pl.col("сумма") > 500)                           # Polars eager
r3 = df_pl.lazy().filter(pl.col("сумма") > 500).collect()          # Polars lazy
print(len(r1), len(r2), len(r3))

In [ ]:
# ДОБАВЛЕНИЕ СТОЛБЦА
df_pd["наценка"] = df_pd["сумма"] * 0.1                                         # pandas (на месте)
df_pl = df_pl.with_columns((pl.col("сумма") * 0.1).alias("наценка"))             # Polars (новый DataFrame)
df_pl.head(3)

In [ ]:
# ГРУППИРОВКА
# pandas:
display(df_pd.groupby("категория")["сумма"].agg(["mean", "sum"]))

# Polars:
df_pl.group_by("категория").agg([
    pl.col("сумма").mean().alias("ср_сумма"),
    pl.col("сумма").sum().alias("итого"),
]).sort("категория")

In [ ]:
# ВЫБОР СТОЛБЦОВ
display(df_pd[["регион", "сумма"]].head(3))   # pandas
df_pl.select(["регион", "сумма"]).head(3)      # Polars

### Join и groupby в Polars

In [ ]:
df = pl.read_csv("../data/sales.csv")

res = (
    df.group_by(["регион", "категория"])
    .agg([
        pl.col("сумма").sum().alias("выручка"),
        pl.col("сумма").mean().alias("ср_чек"),
        pl.col("сумма").count().alias("кол_сделок"),
        pl.col("сумма").max().alias("макс_чек"),
        pl.col("клиент").n_unique().alias("уник_клиентов"),
    ])
    .sort("выручка", descending=True)
)
res.head(10)

### Join в Polars

In [ ]:
transactions = pl.DataFrame({
    "id":        [1, 2, 3, 4],
    "клиент_id": [101, 102, 101, 103],
    "сумма":     [250, 1200, 89, 500],
})

clients = pl.DataFrame({
    "клиент_id": [101, 102, 103],
    "имя":       ["Иванов", "Петрова", "Сидоров"],
    "сегмент":   ["VIP", "Стандарт", "VIP"],
})

# LEFT JOIN
transactions.join(clients, on="клиент_id", how="left")

In [ ]:
# INNER JOIN
transactions.join(clients, on="клиент_id", how="inner")

In [ ]:
# JOIN с разными именами ключей: в таблице клиентов ключ называется "id"
clients_2 = клиенты.rename({"клиент_id": "id"})

transactions.join(
    clients_2,
    left_on="клиент_id",
    right_on="id",
    how="left",
)

### Производительность: pandas vs Polars (5 млн строк, группировка по двум ключам)

In [ ]:
n = 5_000_000
rng = np.random.default_rng(1)
data = {
    "значение":  rng.standard_normal(n),
    "категория": rng.choice(["A", "B", "C", "D"], n),
    "регион":    rng.choice(["Север", "Юг", "Запад", "Восток"], n),
}

# pandas
df_pd = pd.DataFrame(data)
t0 = time.perf_counter()
df_pd.groupby(["категория", "регион"])["значение"].agg(["mean", "std"])
t_pd = time.perf_counter() - t0
print(f"pandas:  {t_pd:.3f}с")

# Polars eager
df_pl = pl.DataFrame(data)
t0 = time.perf_counter()
df_pl.group_by(["категория", "регион"]).agg([
    pl.col("значение").mean().alias("mean"),   # alias обязателен: иначе два столбца
    pl.col("значение").std().alias("std"),     # с одним именем → DuplicateError
])
t_pl = time.perf_counter() - t0
print(f"Polars:  {t_pl:.3f}с  (ускорение ×{t_pd / t_pl:.1f})")

### Типичные ловушки при переходе с pandas

In [ ]:
df = pl.DataFrame({"категория": ["a", "b", "a", "c"], "сумма": [10, 20, 30, 40]})

# 1. Polars неизменяем - нельзя менять «на месте»
# Присваивание df["новый"] = ... не является идиомой Polars (может вызвать ошибку).
# ПРАВИЛЬНО:
df = df.with_columns((pl.col("сумма") * 2).alias("новый_столбец"))
df

In [ ]:
# 3. group_by не гарантирует порядок строк (в отличие от pandas)
print(df.group_by("категория").agg(pl.col("сумма").sum()))   # порядок может меняться между запусками

# Правильно - явная сортировка (или group_by(..., maintain_order=True))
df.group_by("категория").agg(pl.col("сумма").sum()).sort("категория")

In [ ]:
# 4. Индекса строк нет
# pandas:  df.loc["DE"]  → строка по метке индекса
# Polars:  всегда явный фильтр
countries = pl.DataFrame({"страна": ["Германия", "Франция"], "ввп": [4.07, 2.78]})
countries.filter(pl.col("страна") == "Германия")